# Assistants API - 파일 검색

- 파일 검색은 독점 제품 정보나 사용자가 제공한 문서 등 모델 외부의 지식으로 도우미를 강화합니다. OpenAI는 자동으로 문서를 구문 분석 및 청크하고, 임베딩을 생성 및 저장하며, 벡터 및 키워드 검색을 모두 사용하여 관련 콘텐츠를 검색하여 사용자 쿼리에 응답합니다.  

- OpenAI의 Assistant에서 사용하는 파일 검색 기능은 RAG (Retrieval-Augmented Generation) 개념을 구현한 것입니다.

- 작동 방식
    - 문서 구문 분석 및 청크 처리: 파일을 작은 청크로 나누어 각 청크에 대한 임베딩을 생성합니다.
    - 임베딩 생성 및 벡터 저장: 각 청크를 벡터화하여 검색을 위한 벡터 데이터베이스에 저장합니다.
    - 벡터 및 키워드 검색: 사용자 쿼리와 관련된 벡터를 검색하거나 키워드 기반으로 관련 청크를 찾습니다.
    - 응답 생성: 검색된 청크를 바탕으로 사용자 질문에 대한 응답을 생성합니다.
  
이 과정에서 벡터 검색을 통해 관련 정보를 효과적으로 검색하고, 검색된 정보를 생성 과정에 결합하여 응답하므로, 이는 전형적인 RAG 아키텍처의 특징을 따릅니다.

이 실습에서는 회사의 재무제표에 대한 질문에 답변하는 데 도움이 되는  assistant를 만듭니다.

In [1]:
import os
import openai
import sys
sys.path.append('./')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

### 1단계: 파일 검색이 활성화된 새 assistant 만들기
Assistants tools parameter수에서 file_search가 활성화된 새 assistant를 만듭니다  
file_search 도구가 활성화되면 모델은 사용자 메시지를 기반으로 콘텐츠를 검색할 시기를 결정합니다.

In [2]:
from openai import OpenAI
client = OpenAI()

Model = "gpt-4o-mini"

In [3]:
assistant = client.beta.assistants.create(
  name="Financial Analyst Assistant",
  instructions="당신은 전문 재무 분석가입니다. 지식 기반을 사용하여 감사된 재무제표에 대한 질문에 답하십시오.",
  model=Model,
  tools=[{"type": "file_search"}],
)

assistant

Assistant(id='asst_OO0iSwRlVljyNLX6GhCggeOn', created_at=1730532019, description=None, instructions='당신은 전문 재무 분석가입니다. 지식 기반을 사용하여 감사된 재무제표에 대한 질문에 답하십시오.', metadata={}, model='gpt-4o-mini', name='Financial Analyst Assistant', object='assistant', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21')))], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=None, file_search=ToolResourcesFileSearch(vector_store_ids=[])), top_p=1.0)

### 2단계: Vector Store 생성

- 가격: FILE SEARCH 기능은 벡터 저장소(storage) 용량을 기준으로 하루에 $0.10/GB의 요금이 부과됩니다. 첫 번째 1GB는 무료로 제공됩니다​.
- 보존기간: 생성된 벡터 저장소가 일주일 동안 비활성 상태로 남아 있으면 자동으로 삭제됩니다.

In [4]:
# "Financial Statements"라는 벡터 스토어 생성
vector_store = client.beta.vector_stores.create(name="Financial Statements")
 
# OpenAI에 업로드할 파일 준비
file_paths = ["edgar/goog-10k.pdf", "edgar/brka-10k.pdf"]
file_streams = [open(path, "rb") for path in file_paths]
 
# 업로드 및 폴링 SDK 도우미를 사용하여 파일을 업로드하고 벡터 스토어에 추가,
# 파일 배치의 완료 상태를 폴링합니다.
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)
 
# 이 작업의 결과를 보기 위해 상태 및 파일 개수를 출력할 수 있습니다.
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2)


### 3단계: 새로운 Vector Store를 사용하도록 어시스턴트 업데이트
어시스턴트가 파일에 액세스할 수 있도록 하려면 어시스턴트의 tool_resources를 새 vector_store ID로 업데이트합니다.

In [5]:
assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)
assistant

Assistant(id='asst_OO0iSwRlVljyNLX6GhCggeOn', created_at=1730532019, description=None, instructions='당신은 전문 재무 분석가입니다. 지식 기반을 사용하여 감사된 재무제표에 대한 질문에 답하십시오.', metadata={}, model='gpt-4o-mini', name='Financial Analyst Assistant', object='assistant', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21')))], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=None, file_search=ToolResourcesFileSearch(vector_store_ids=['vs_i03pTWGd4oARRVwaYNh2auJB'])), top_p=1.0)

### 4단계: 스레드 만들기
스레드에 메시지 첨부 파일로 파일을 첨부할 수도 있습니다. 이렇게 하면 스레드와 연결된 또 다른 벡터 저장소가 생성됩니다. 또는 이 스레드에 이미 연결된 벡터 저장소가 있는 경우 새 파일을 기존 스레드 벡터 저장소에 연결합니다. 이 스레드에서 실행을 생성하면 파일 검색 도구는 어시스턴트의 vector_store와 스레드의 vector_store를 모두 쿼리합니다.  

여기서 사용자는 Apple의 최신 10-K 파일을 추가로 첨부합니다. 

In [11]:
# OpenAI에 사용자 제공 파일 업로드
message_file = client.files.create(
  file=open("edgar/aapl-10k.pdf", "rb"), purpose="assistants"
)

# 스레드 생성 및 파일을 메시지에 첨부
thread = client.beta.threads.create(
  messages=[
    {
      "role": "user",
      "content": "2023년 10월 말에 AAPL의 발행 주식 수는 얼마였나요?",
      # 새 파일을 메시지에 첨부합니다.
      "attachments": [
        { "file_id": message_file.id, 
          "tools": [{"type": "file_search"}] }
      ],
    }
  ]
)

# 이제 스레드에는 파일이 있는 벡터 저장소가 툴 리소스로 있습니다.
print(thread.tool_resources.file_search)

ToolResourcesFileSearch(vector_store_ids=['vs_ly1bT92g3sXwSY6vHLpxwsEr'])


### 5단계: Run 만들기 및 출력 확인
이제Run행을 생성하고 모델이 파일 검색 도구를 사용하여 사용자의 질문에 대한 응답을 제공하는 것을 관찰합니다.

In [12]:
# `create_and_poll` 메서드를 사용하여 새로운 실행(run)을 생성하고 해당 실행이 완료될 때까지 폴링합니다.
run = client.beta.threads.runs.create_and_poll(
  thread_id=thread.id,  # 실행할 스레드의 ID
  assistant_id=assistant.id,  # 실행할 어시스턴트의 ID
  instructions="사용자를 고객님이라고 부르세요. 이 사용자는 프리미엄 계정을 가지고 있습니다.",   # 어시스턴트에게 제공할 지시사항
)

In [13]:
# 실행 상태 확인
if run.status == 'completed': 
  messages = client.beta.threads.messages.list(
    thread_id=thread.id
  )
  print(messages)
else:
  print(run.status)

SyncCursorPage[Message](data=[Message(id='msg_BRUg8nlCaFd5pucM1wUzZVrP', assistant_id='asst_OO0iSwRlVljyNLX6GhCggeOn', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[FileCitationAnnotation(end_index=60, file_citation=FileCitation(file_id='file-ct1ykkZ5jtSMiUFD0XYdHuRF'), start_index=47, text='【4:11†source】', type='file_citation')], value='2023년 10월 말 기준으로 AAPL의 발행 주식 수는 15,943,425 주입니다【4:11†source】.'), type='text')], created_at=1730532410, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_hZnM4T8s5zzS9d2RkNvjtcDU', status=None, thread_id='thread_xhyn6Lanz96OZdFtp7ApE2an'), Message(id='msg_7m2Yh8Y8OjUxcUSzhRGZrQBO', assistant_id=None, attachments=[Attachment(file_id='file-ct1ykkZ5jtSMiUFD0XYdHuRF', tools=[AttachmentToolAssistantToolsFileSearchTypeOnly(type='file_search')])], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='2023년 10월 말에 AAPL의 발행 주식 수는 얼마였나요?'),

In [14]:
messages.data[0].content[0].text.value

'2023년 10월 말 기준으로 AAPL의 발행 주식 수는 15,943,425 주입니다【4:11†source】.'

### Thread에 새로운 message 추가 및 Run 생성

In [15]:
message = client.beta.threads.messages.create(
      thread_id=thread.id,
      role="user",
      content="2023년 Google의 당기 순이익은 얼마였나요? BERKSHIRE HATHAWAY INC. 와 어느 쪽의 당기 순이익이 더 많았나요?"
    )

In [16]:
# `create_and_poll` 메서드를 사용하여 새로운 실행(run)을 생성하고 해당 실행이 완료될 때까지 폴링합니다.
run = client.beta.threads.runs.create_and_poll(
  thread_id=thread.id,  # 실행할 스레드의 ID
  assistant_id=assistant.id,  # 실행할 어시스턴트의 ID
  instructions="사용자를 고객님이라고 부르세요. 이 사용자는 프리미엄 계정을 가지고 있습니다.",   # 어시스턴트에게 제공할 지시사항
)

In [17]:
# 실행 상태 확인
if run.status == 'completed': 
  messages = client.beta.threads.messages.list(
    thread_id=thread.id
  )
  print(messages)
else:
  print(run.status)

SyncCursorPage[Message](data=[Message(id='msg_S9CMG0dNDEGUvDHAuHKtWzHT', assistant_id='asst_OO0iSwRlVljyNLX6GhCggeOn', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[FileCitationAnnotation(end_index=65, file_citation=FileCitation(file_id='file-X3fl7qWsjsDjBaI38D4ouNxj'), start_index=52, text='【14:3†source】', type='file_citation'), FileCitationAnnotation(end_index=147, file_citation=FileCitation(file_id='file-MsV6YYTxAAs4IrUmwUSVFlDi'), start_index=134, text='【14:9†source】', type='file_citation')], value='2023년 Google의 당기 순이익은 69,170백만 달러(69.2억 달러)로 보고되었습니다【14:3†source】. 반면, Berkshire Hathaway의 2023년 당기 순이익은 42,521백만 달러(42.5억 달러)로 나타났습니다【14:9†source】.\n\n따라서 Google의 당기 순이익이 Berkshire Hathaway보다 더 많습니다.'), type='text')], created_at=1730532582, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_mlbjdrbi29xcKS33BsSzZCt5', status=None, thread_id='thread_xhyn6Lanz96OZdFtp7ApE2an'), Message(id='

In [18]:
messages.data[0].content[0].text.value

'2023년 Google의 당기 순이익은 69,170백만 달러(69.2억 달러)로 보고되었습니다【14:3†source】. 반면, Berkshire Hathaway의 2023년 당기 순이익은 42,521백만 달러(42.5억 달러)로 나타났습니다【14:9†source】.\n\n따라서 Google의 당기 순이익이 Berkshire Hathaway보다 더 많습니다.'

In [19]:
# messages 객체의 데이터에서 role과 content 값을 추출하여 리스트에 저장
values = [(messages.data[i].role, messages.data[i].content[0].text.value) for i in range(len(messages.data)-1, -1, -1)]

# 추출한 값들을 출력
for role, value in values:
    print(f"role: {role}")  # 역할(role) 출력
    print(value)  # 내용(value) 출력
    print(150*"-")  # 구분선 출력

role: user
2023년 10월 말에 AAPL의 발행 주식 수는 얼마였나요?
------------------------------------------------------------------------------------------------------------------------------------------------------
role: assistant
2023년 10월 말 기준으로 AAPL의 발행 주식 수는 15,943,425 주입니다【4:11†source】.
------------------------------------------------------------------------------------------------------------------------------------------------------
role: user
2023년 Google의 당기 순이익은 얼마였나요? BERKSHIRE HATHAWAY INC. 와 어느 쪽의 당기 순이익이 더 많았나요?
------------------------------------------------------------------------------------------------------------------------------------------------------
role: assistant
2023년 Google의 당기 순이익은 69,170백만 달러(69.2억 달러)로 보고되었습니다【14:3†source】. 반면, Berkshire Hathaway의 2023년 당기 순이익은 42,521백만 달러(42.5억 달러)로 나타났습니다【14:9†source】.

따라서 Google의 당기 순이익이 Berkshire Hathaway보다 더 많습니다.
------------------------------------------------------------------------------------------------------------------------

## 실습 : Prompt 수정하여 실행